Prepara la base `panteon_exp` y crea las colecciones para comparar la Estrategia A (`sequences_a`, con k-mers embebidos) y la Estrategia B (`sequences_light` sin k-mers + `sequence_kmers` aparte):

In [1]:
import sys
import time
from copy import deepcopy
from pathlib import Path
import pandas as pd
from pymongo import MongoClient, ASCENDING

sys.path.insert(0, str(Path("..").resolve()))

from database.setup_db import SCHEMA_SPECIES, SCHEMA_SEQUENCES
from pipeline.limpiar import limpiar_csv
from pipeline.fasta import leer_fasta
import pipeline.carga as pc

FASTA = "../data/PanTEon_Database_v1.6.1.fasta"
CSV = "../data/PanTEon_Database_metadata_v1.6.1.csv"
N_MUESTRA = 20000
TAM = 1000

client = MongoClient("mongodb://localhost:27017")
db = client["panteon_exp"]

esquema_light = deepcopy(SCHEMA_SEQUENCES)
esquema_light["$jsonSchema"]["required"].remove("kmers")
esquema_light["$jsonSchema"]["properties"].pop("kmers", None)

esquema_kmers = {
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["sequence_id", "kmers"],
        "properties": {
            "sequence_id": {"bsonType": "string", "minLength": 1},
            "kmers": SCHEMA_SEQUENCES["$jsonSchema"]["properties"]["kmers"],
        },
    }
}

for nombre, esq in [("species", SCHEMA_SPECIES), ("sequences_a", SCHEMA_SEQUENCES),
                    ("sequences_light", esquema_light), ("sequence_kmers", esquema_kmers)]:
    if nombre in db.list_collection_names():
        db[nombre].drop()
    db.create_collection(nombre, validator=esq, validationLevel="strict", validationAction="error")

db.species.create_index([("species_key", ASCENDING)], unique=True, name="idx_species_key_unique")
db.species.create_index([("Kingdom", ASCENDING), ("Class", ASCENDING)], name="idx_kingdom_class")
db.species.create_index([("Source", ASCENDING)], name="idx_species_source")

for col in (db.sequences_a, db.sequences_light):
    col.create_index([("sequence_id", ASCENDING)], unique=True, name="idx_sequence_id_unique")
    col.create_index([("species_key", ASCENDING), ("te_superfamily", ASCENDING)], name="idx_species_superfamily")
    col.create_index([("te_class", ASCENDING), ("te_order", ASCENDING), ("te_superfamily", ASCENDING)], name="idx_te_hierarchy")
    col.create_index([("metrics.length", ASCENDING)], name="idx_metrics_length")
    col.create_index([("metrics.gc_content", ASCENDING)], name="idx_metrics_gc")

db.sequence_kmers.create_index([("sequence_id", ASCENDING)], unique=True, name="idx_sequence_id_unique")

tabla = limpiar_csv(CSV)
pc.cargar_lotes(db.species, tabla.to_dict("records"), "species_key")
print("species:", db.species.count_documents({}), "| colecciones:", db.list_collection_names())

species: 2790 | colecciones: ['species', 'sequences_light', 'sequences_a', 'sequence_kmers']


Carga 20.000 secuencias por lotes de 1.000 midiendo el tiempo de inserción en cada estrategia con `time.perf_counter()`:

In [2]:
t_calc, t_a, t_light, t_kmers = 0.0, 0.0, 0.0, 0.0
lote_a, lote_light, lote_kmers = [], [], []
n = 0

for campos in leer_fasta(FASTA):
    t0 = time.perf_counter()
    doc = pc.armar_secuencia(campos)
    doc_k = {"sequence_id": doc["sequence_id"], "kmers": doc["kmers"]}
    doc_l = {k: v for k, v in doc.items() if k != "kmers"}
    t_calc += time.perf_counter() - t0

    lote_a.append(doc)
    lote_light.append(doc_l)
    lote_kmers.append(doc_k)
    n += 1

    if len(lote_a) >= TAM:
        t0 = time.perf_counter()
        db.sequences_a.insert_many([d.copy() for d in lote_a], ordered=False)
        t_a += time.perf_counter() - t0

        t0 = time.perf_counter()
        db.sequences_light.insert_many([d.copy() for d in lote_light], ordered=False)
        t_light += time.perf_counter() - t0

        t0 = time.perf_counter()
        db.sequence_kmers.insert_many([d.copy() for d in lote_kmers], ordered=False)
        t_kmers += time.perf_counter() - t0

        lote_a.clear(); lote_light.clear(); lote_kmers.clear()

    if n >= N_MUESTRA:
        break

client.admin.command("fsync")
t_b = t_light + t_kmers

df_tiempos = pd.DataFrame([
    {"estrategia": "Estrategia A (sequences_a)", "docs": n, "calculo_s": round(t_calc, 2),
     "insercion_s": round(t_a, 2), "total_s": round(t_calc + t_a, 2), "docs_s": round(n / t_a, 1)},
    {"estrategia": "B1: sequences_light", "docs": n, "calculo_s": None,
     "insercion_s": round(t_light, 2), "total_s": None, "docs_s": round(n / t_light, 1)},
    {"estrategia": "B2: sequence_kmers", "docs": n, "calculo_s": None,
     "insercion_s": round(t_kmers, 2), "total_s": None, "docs_s": round(n / t_kmers, 1)},
    {"estrategia": "Estrategia B total (light + kmers)", "docs": n, "calculo_s": round(t_calc, 2),
     "insercion_s": round(t_b, 2), "total_s": round(t_calc + t_b, 2), "docs_s": round(n / t_b, 1)},
])
df_tiempos

estrategia,docs,calculo_s,insercion_s,total_s,docs_s
Estrategia A (sequences_a),20000,12.76,4.75,17.51,4208.8
B1: sequences_light,20000,NaN,0.60,NaN,33408.8
B2: sequence_kmers,20000,NaN,4.20,NaN,4767.0
Estrategia B total (light + kmers),20000,12.76,4.79,17.55,4171.7


Espacio en disco con `collStats` para las 20.000 secuencias:

In [3]:
st_a = db.command("collStats", "sequences_a")
st_l = db.command("collStats", "sequences_light")
st_k = db.command("collStats", "sequence_kmers")

for nombre, st in [("sequences_a", st_a), ("sequences_light", st_l), ("sequence_kmers", st_k)]:
    print(f"{nombre}: {st['count']:,} docs | medio {st['avgObjSize']/1024:.2f} KB | "
          f"sin comprimir {st['size']/1024**2:.2f} MB | en disco {st['storageSize']/1024**2:.2f} MB | "
          f"índices {st['totalIndexSize']/1024**2:.2f} MB")

def resumen_stats(nombre, st):
    return {
        "coleccion": nombre,
        "docs": st["count"],
        "avgObjSize_KB": round(st["avgObjSize"] / 1024, 2),
        "size_MB": round(st["size"] / 1024**2, 2),
        "storageSize_MB": round(st["storageSize"] / 1024**2, 2),
        "totalIndexSize_MB": round(st["totalIndexSize"] / 1024**2, 2),
    }

df_espacio = pd.DataFrame([
    resumen_stats("Estrategia A (sequences_a)", st_a),
    resumen_stats("B1: sequences_light", st_l),
    resumen_stats("B2: sequence_kmers", st_k),
    {
        "coleccion": "Estrategia B total (light + kmers)",
        "docs": st_l["count"],
        "avgObjSize_KB": round((st_l["avgObjSize"] + st_k["avgObjSize"]) / 1024, 2),
        "size_MB": round((st_l["size"] + st_k["size"]) / 1024**2, 2),
        "storageSize_MB": round((st_l["storageSize"] + st_k["storageSize"]) / 1024**2, 2),
        "totalIndexSize_MB": round((st_l["totalIndexSize"] + st_k["totalIndexSize"]) / 1024**2, 2),
    },
])
df_espacio

sequences_a: 20,000 docs | medio 29.13 KB | sin comprimir 568.95 MB | en disco 228.88 MB | índices 1.37 MB
sequences_light: 20,000 docs | medio 2.92 KB | sin comprimir 57.10 MB | en disco 26.63 MB | índices 1.46 MB
sequence_kmers: 20,000 docs | medio 26.26 KB | sin comprimir 512.82 MB | en disco 198.98 MB | índices 0.54 MB


coleccion,docs,avgObjSize_KB,size_MB,storageSize_MB,totalIndexSize_MB
Estrategia A (sequences_a),20000,29.13,568.95,228.88,1.37
B1: sequences_light,20000,2.92,57.10,26.63,1.46
B2: sequence_kmers,20000,26.26,512.82,198.98,0.54
Estrategia B total (light + kmers),20000,29.18,569.93,225.61,2.01


Proyección a las 239.946 secuencias completas frente a los valores medidos en `05_pipeline.ipynb`:

In [4]:
f = 239946 / N_MUESTRA

def proyectar(nombre, size_b, storage_b, idx_b, avg_b):
    return {
        "coleccion": nombre,
        "avgObjSize_KB": round(avg_b / 1024, 2),
        "size_GB": round(size_b * f / 1024**3, 2),
        "storageSize_GB": round(storage_b * f / 1024**3, 2),
        "totalIndexSize_MB": round(idx_b * f / 1024**2, 2),
    }

df_proyeccion = pd.DataFrame([
    {"coleccion": "Estrategia A real (05_pipeline)", "avgObjSize_KB": 36.70,
     "size_GB": 8.40, "storageSize_GB": 3.42, "totalIndexSize_MB": 22.30},
    proyectar("Estrategia A proyectada", st_a["size"], st_a["storageSize"], st_a["totalIndexSize"], st_a["avgObjSize"]),
    proyectar("B1: sequences_light proyectada", st_l["size"], st_l["storageSize"], st_l["totalIndexSize"], st_l["avgObjSize"]),
    proyectar("B2: sequence_kmers proyectada", st_k["size"], st_k["storageSize"], st_k["totalIndexSize"], st_k["avgObjSize"]),
    proyectar("Estrategia B total proyectada",
              st_l["size"] + st_k["size"],
              st_l["storageSize"] + st_k["storageSize"],
              st_l["totalIndexSize"] + st_k["totalIndexSize"],
              st_l["avgObjSize"] + st_k["avgObjSize"]),
])
df_proyeccion

coleccion,avgObjSize_KB,size_GB,storageSize_GB,totalIndexSize_MB
Estrategia A real (05_pipeline),36.70,8.40,3.42,22.30
Estrategia A proyectada,29.13,6.67,2.68,16.45
B1: sequences_light proyectada,2.92,0.67,0.31,17.57
B2: sequence_kmers proyectada,26.26,6.01,2.33,6.51
Estrategia B total proyectada,29.18,6.68,2.64,24.09
